# 1. Instalación de librerías

In [2]:
!pip install -q decord transformers evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 125.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.0 MB/s eta 0:00:00


In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from decord import VideoReader, cpu
from tqdm import tqdm
from datasets import Dataset

# 2. Configuración y rutas

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# MODELO
MODEL_CHECKPOINT = "MCG-NJU/videomae-base"

# RUTAS
RUTA_BASE_VIDEOS = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/"
CSV_TRAIN_MASTER = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/EXIST2025_training_3_1_master.csv"
CSV_TEST = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/EXIST2025_test_3_1.csv"

# 3. Carga de datos

In [ ]:
print("Cargando particiones fijas...")
df_train_master = pd.read_csv(CSV_TRAIN_MASTER)
df_test = pd.read_csv(CSV_TEST)

Cargando particiones fijas...


In [ ]:
# Limpieza de índices y etiquetas
for df in [df_train_master, df_test]:
    if "Unnamed: 0" in df.columns:
        df.drop(columns=["Unnamed: 0"], inplace=True)
    if "label_task_3_1_merged" in df.columns:
        df.rename(columns={"label_task_3_1_merged": "label"}, inplace=True)
    df["label"] = df["label"].astype(int)

In [ ]:
# División Dinámica (90/10) del Train Master
train_df, val_df = train_test_split(
    df_train_master,
    test_size=0.10,
    stratify=df_train_master["label"],
    random_state=42
)

In [ ]:
# Convertir las rutas relativas en rutas absolutas completas para decord
def fix_video_paths(df, base_path):
    # Asume que el CSV tiene una columna 'path_video' o 'id_EXIST' + '.mp4'
    # Ajusta esto según el nombre real de tu columna en el CSV de texto
    columna = 'path_video' if 'path_video' in df.columns else 'id_EXIST'

    rutas = []
    for val in df[columna]:
        if not str(val).endswith(".mp4"):
            val = str(val) + ".mp4"

        # IMPORTANTE: A veces el CSV tiene "videos/nombre.mp4".
        ruta_completa = os.path.join(base_path, val) # Ajusta "videos" si es necesario
        rutas.append(ruta_completa)

    df['ruta_absoluta'] = rutas
    return df

In [ ]:
train_df = fix_video_paths(train_df.copy(), RUTA_BASE_VIDEOS)
val_df = fix_video_paths(val_df.copy(), RUTA_BASE_VIDEOS)

In [ ]:
print(f"\nVídeos en Train: {len(train_df)}")
print(f"Vídeos en Valid: {len(val_df)}")
print("Ejemplo de ruta:", train_df['ruta_absoluta'].iloc[0])


Vídeos en Train: 1805
Vídeos en Valid: 201
Ejemplo de ruta: /content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/videos/7133053554763287813.mp4


# 4. Función central de extracción

In [ ]:
def sample_frame_indices(clip_len, total_frames):
    if total_frames <= clip_len:
        # Si el vídeo es súper corto, repetimos frames
        return np.linspace(0, total_frames - 1, num=clip_len, dtype=int).tolist()
    else:
        # Muestreo uniforme a lo largo de todo el vídeo
        indices = np.linspace(0, total_frames - 1, num=clip_len, dtype=int)
        return indices.tolist()

# 5. Preparación del dataset de vídeo (pytorch)

In [ ]:
from transformers import VideoMAEImageProcessor
from torch.utils.data import Dataset
import decord
from decord import VideoReader, cpu

In [ ]:
# Silenciamos los logs de decord para que no ensucien la consola
decord.bridge.set_bridge('torch')

In [ ]:
# Cargamos el procesador específico de VideoMAE
print("Cargando VideoMAEImageProcessor...")
processor = VideoMAEImageProcessor.from_pretrained(MODEL_CHECKPOINT)

Cargando VideoMAEImageProcessor...


preprocessor_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

In [ ]:
class VideoClassificationDataset(Dataset):
    def __init__(self, df, processor, clip_len=16):
        """
        df: DataFrame que contiene 'ruta_absoluta' y 'label'
        processor: VideoMAEImageProcessor
        clip_len: Número de frames que espera el modelo (VideoMAE usa 16)
        """
        self.rutas = df['ruta_absoluta'].tolist()
        self.etiquetas = df['label'].tolist()
        self.processor = processor
        self.clip_len = clip_len

    def __len__(self):
        return len(self.rutas)

    def __getitem__(self, idx):
        ruta_video = self.rutas[idx]
        etiqueta = self.etiquetas[idx]

        # 1. Leemos el vídeo de forma súper eficiente con Decord (solo en CPU para no bloquear la GPU)
        try:
            # ctx=cpu(0) es importante para evitar cuelgues raros en Colab
            vr = VideoReader(ruta_video, ctx=cpu(0))
            total_frames = len(vr)

            # 2. Obtenemos los índices de los 16 frames repartidos uniformemente
            frame_indices = sample_frame_indices(self.clip_len, total_frames)

            # 3. Extraemos SOLO esos 16 frames (devuelve un tensor de PyTorch gracias al bridge)
            # El formato original suele ser [T, H, W, C] (Tiempo, Alto, Ancho, Canales)
            frames = vr.get_batch(frame_indices).numpy()

            # 4. Convertimos la lista de numpy arrays al formato que quiere Hugging Face
            # El procesador espera una lista de arrays numpy o un solo tensor gigante.
            # Le pasamos la lista de frames y él se encarga de redimensionar a 224x224
            inputs = self.processor(list(frames), return_tensors="pt")

            # El procesador devuelve un diccionario {'pixel_values': tensor_gigante}
            # Extraemos el tensor y le quitamos la dimensión extra del 'batch' que pone por defecto
            pixel_values = inputs["pixel_values"][0]

        except Exception as e:
            # A veces un .mp4 puede estar corrupto.
            # Si falla, imprimimos el error y devolvemos un tensor vacío pero con las dimensiones correctas
            # para que el entrenamiento no se corte (es un salvavidas).
            print(f"\n⚠️ Error leyendo {ruta_video}: {e}")
            import torch
            # CORREGIDO: El orden correcto para VideoMAE es (Frames, Canales, Alto, Ancho)
            pixel_values = torch.zeros((self.clip_len, 3, 224, 224))
            etiqueta = 0 # Valor por defecto

        # Devolvemos el tensor de 4D del vídeo y su etiqueta
        return {"pixel_values": pixel_values, "label": etiqueta}

# 6. Inicialización de los datasets

In [ ]:
print("Construyendo los Datasets de PyTorch...")

# Usamos los DataFrames que ya tienen la columna 'ruta_absoluta'
train_dataset = VideoClassificationDataset(train_df, processor, clip_len=16)
valid_dataset = VideoClassificationDataset(val_df, processor, clip_len=16)

Construyendo los Datasets de PyTorch...


In [ ]:
# Probamos que funciona correctamente sacando el primer elemento
print("\nComprobando el primer vídeo del dataset (esto extraerá 16 frames)...")
ejemplo = train_dataset[0]
print("Dimensiones del tensor final (Debe ser [Canales(3), Frames(16), Alto(224), Ancho(224)]):")
print(ejemplo["pixel_values"].shape)
print("Etiqueta:", ejemplo["label"])


Comprobando el primer vídeo del dataset (esto extraerá 16 frames)...
Dimensiones del tensor final (Debe ser [Canales(3), Frames(16), Alto(224), Ancho(224)]):
torch.Size([16, 3, 224, 224])
Etiqueta: 0


# 7. Carga de modelo y métricas

In [ ]:
from transformers import VideoMAEForVideoClassification
import evaluate
import numpy as np
import torch

print("\nCargando modelo VideoMAEForVideoClassification...")
# Mapeos de etiquetas
id2label = {0: "No Misógino", 1: "Misógino"}
label2id = {"No Misógino": 0, "Misógino": 1}

# Cargamos el modelo pre-entrenado, pero le cambiamos la "cabeza" final
# para que tenga solo 2 neuronas de salida (nuestras 2 clases).
# ignore_mismatched_sizes=True es vital porque el modelo original de HF estaba
# entrenado para 400 clases (Kinetics-400), y nosotros lo forzamos a 2.
model = VideoMAEForVideoClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)


Cargando modelo VideoMAEForVideoClassification...


config.json:   0%|          | 0.00/725 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  377MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/160 [00:00<?, ?it/s]

[transformers] VideoMAEForVideoClassification LOAD REPORT from: MCG-NJU/videomae-base
Key                                                                  | Status     | 
---------------------------------------------------------------------+------------+-
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.value.weight | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.output.dense.weight              | UNEXPECTED | 
videomae.encoder.layer.{0...11}.attention.attention.q_bias           | UNEXPECTED | 
videomae.encoder.layer.{0...11}.attention.attention.v_bias           | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.key.weight   | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.query.weight | UNEXPECTED | 
mask_token                                                           | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.layernorm_before.bias            | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.v_bias  

In [ ]:
# Definimos cómo evaluar (Usaremos F1-Macro y Accuracy, el estándar del TFM)
f1_metric = evaluate.load("f1")
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)

    f1 = f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"]
    acc = accuracy_metric.compute(predictions=preds, references=labels)["accuracy"]

    return {"f1": f1, "accuracy": acc}

# 8. Configuración del entrenamiento (Trainer)

In [ ]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
import os

In [ ]:
OUTPUT_DIR = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/Vision/VideoMAE_FineTuned"
os.makedirs(OUTPUT_DIR, exist_ok=True)

batch_size = 4

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    remove_unused_columns=False, # ¡CRUCIAL PARA VÍDEO! Si es True, HF borra la columna "pixel_values"
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,          # LR típico para fine-tuning de VideoMAE
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    gradient_accumulation_steps=2, # Simulamos un batch de 8 para mejor convergencia
    num_train_epochs=5,          # Le damos 5 épocas (Early Stopping lo parará si es necesario)
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_strategy="steps",
    logging_steps=50,
    fp16=True,                   # Aceleración por hardware obligatoria en vídeo
    report_to="none",            # Apagamos wandb/logs externos
    dataloader_num_workers=2,    # Usa 2 procesos en segundo plano para leer los .mp4 más rápido
    dataloader_pin_memory=True
)

In [ ]:
# Inicializamos el Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)] # Si no mejora en 2 épocas, para
)

In [ ]:
print("\n🚀 Lanzando el entrenamiento de VideoMAE (Ponte cómodo, esto tomará un rato)...")
trainer.train()

print("\n💾 Guardando el modelo definitivo...")
trainer.save_model(os.path.join(OUTPUT_DIR, "modelo_final"))
# También guardamos el procesador para que la inferencia sea fácil luego
processor.save_pretrained(os.path.join(OUTPUT_DIR, "modelo_final"))

print("✅ ¡Entrenamiento completado y guardado con éxito!")


🚀 Lanzando el entrenamiento de VideoMAE (Ponte cómodo, esto tomará un rato)...


Epoch,Training Loss,Validation Loss,F1,Accuracy
1,0.694119,0.687390,0.528612,0.552239
2,0.654894,0.727860,0.497513,0.517413
3,0.535342,0.872737,0.523542,0.527363



⚠️ Error leyendo /content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/videos/7136934291056905477.mp4: [08:26:45] /github/workspace/src/video/video_reader.cc:486: Error: av_read_frame failed with 1094995529

⚠️ Error leyendo /content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/videos/6910788530201529605.mp4: [08:28:20] /github/workspace/src/video/video_reader.cc:486: Error: av_read_frame failed with 1094995529

⚠️ Error leyendo /content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/videos/6932866433571376390.mp4: [08:29:32] /github/workspace/src/video/video_reader.cc:486: Error: av_read_frame failed with 1094995529

⚠️ Error leyendo /content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/videos/7089505854399073578.mp4: [08:29:41] /github/workspace/src/video/ffmpeg/threaded_decoder.cc:292: [08:29:41] /github/workspace/src/video/ffmpeg/filter_graph.cc:100: Check failed: av_buffersrc_add_frame_flags(buffersrc_ctx_, frame, AV_BUFFERSRC_FLAG_KE

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


⚠️ Error leyendo /content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/videos/6910093785212980482.mp4: [08:51:58] /github/workspace/src/video/video_reader.cc:486: Error: av_read_frame failed with 1094995529

⚠️ Error leyendo /content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/videos/6979380675371650310.mp4: [08:52:10] /github/workspace/src/video/video_reader.cc:486: Error: av_read_frame failed with 1094995529

⚠️ Error leyendo /content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/videos/6855976112825142533.mp4: [08:52:56] /github/workspace/src/video/video_reader.cc:486: Error: av_read_frame failed with 1094995529

⚠️ Error leyendo /content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/videos/6979707057066888454.mp4: [08:53:30] /github/workspace/src/video/ffmpeg/threaded_decoder.cc:292: [08:53:30] /github/workspace/src/video/ffmpeg/filter_graph.cc:100: Check failed: av_buffersrc_add_frame_flags(buffersrc_ctx_, frame, AV_BUFFERSRC_FLAG_KE

...


⚠️ Error leyendo /content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/videos/6960334600639991046.mp4: [09:01:14] /github/workspace/src/video/video_reader.cc:486: Error: av_read_frame failed with 1094995529

⚠️ Error leyendo /content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/videos/6979380675371650310.mp4: [09:01:57] /github/workspace/src/video/video_reader.cc:486: Error: av_read_frame failed with 1094995529

⚠️ Error leyendo /content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/videos/7010961027382725894.mp4: [09:02:07] /github/workspace/src/video/video_reader.cc:486: Error: av_read_frame failed with 1094995529

⚠️ Error leyendo /content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/videos/6968610667347578117.mp4: [09:02:53] /github/workspace/src/video/ffmpeg/threaded_decoder.cc:292: [09:02:53] /github/workspace/src/video/ffmpeg/threaded_decoder.cc:198: Check failed: avcodec_send_packet(dec_ctx_.get(), __null) >= 0 (-1094995529 vs. 0)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


💾 Guardando el modelo definitivo...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ ¡Entrenamiento completado y guardado con éxito!


# Resultados contra fichero de Test

In [ ]:
import os
import torch
import pandas as pd
import numpy as np
from decord import VideoReader, cpu
from tqdm import tqdm
from transformers import VideoMAEImageProcessor, VideoMAEForVideoClassification
from sklearn.metrics import classification_report, f1_score, accuracy_score, confusion_matrix

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Configuración y rutas

In [ ]:
RUTA_BASE_VIDEOS = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/"
CSV_TEST = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/EXIST2025_test_3_1.csv"
MODEL_DIR = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/Vision/VideoMAE_FineTuned/modelo_final"
CSV_SALIDA = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/Vision/predicciones_videomae_test.csv"

## 2. Carga de los datos (test) y del modelo

In [ ]:
print("Cargando el dataset estático de test...")
test_df = pd.read_csv(CSV_TEST)

# Limpieza de columnas igual que en Train
if "Unnamed: 0" in test_df.columns:
    test_df.drop(columns=["Unnamed: 0"], inplace=True)
if "label_task_3_1_merged" in test_df.columns:
    test_df.rename(columns={"label_task_3_1_merged": "label"}, inplace=True)
test_df["label"] = test_df["label"].astype(int)

Cargando el dataset estático de test...


In [ ]:
# Arreglar rutas absolutas (usamos la misma lógica que tenías)
def fix_video_paths(df, base_path):
    columna = 'path_video' if 'path_video' in df.columns else 'id_EXIST'
    rutas = []
    for val in df[columna]:
        if not str(val).endswith(".mp4"): val = str(val) + ".mp4"
        rutas.append(os.path.join(base_path, val))
    df['ruta_absoluta'] = rutas
    return df

test_df = fix_video_paths(test_df, RUTA_BASE_VIDEOS)

In [ ]:
# Función de muestreo de frames (se usa en el entrenamiento)
def sample_frame_indices(clip_len, total_frames):
    if total_frames <= clip_len:
        return np.linspace(0, total_frames - 1, num=clip_len, dtype=int).tolist()
    else:
        return np.linspace(0, total_frames - 1, num=clip_len, dtype=int).tolist()

In [ ]:
print("Cargando procesador y modelo VideoMAE entrenado...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
processor = VideoMAEImageProcessor.from_pretrained(MODEL_DIR)
model = VideoMAEForVideoClassification.from_pretrained(MODEL_DIR).to(device)
model.eval()

print(f"Iniciando inferencia sobre {len(test_df)} vídeos de test...")

Cargando procesador y modelo VideoMAE entrenado...


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

Iniciando inferencia sobre 502 vídeos de test...


## 3. Inferencia directa de vídeo

In [ ]:
y_true = []
y_pred = []
resultados_para_csv = []

for index, row in tqdm(test_df.iterrows(), total=len(test_df)):
    id_vid = row['id_EXIST']
    ruta_video = row['ruta_absoluta']
    true_label = row['label']

    try:
        # Lectura eficiente con Decord
        vr = VideoReader(ruta_video, ctx=cpu(0))
        total_frames = len(vr)
        frame_indices = sample_frame_indices(16, total_frames)
        frames = vr.get_batch(frame_indices).asnumpy()

        # Procesamos los 16 fotogramas de golpe
        inputs = processor(list(frames), return_tensors="pt")
        pixel_values = inputs["pixel_values"].to(device)

        # Inferencia
        with torch.no_grad():
            outputs = model(pixel_values=pixel_values)
            probs = torch.nn.functional.softmax(outputs.logits, dim=-1)[0]
            prob_misogino = probs[1].item()

    except Exception as e:
        print(f"\n⚠️ Error procesando {ruta_video}: {e}")
        # En caso de vídeo corrupto, asumimos incertidumbre (0.5) para no romper el test
        prob_misogino = 0.5

    # 1 si prob > 0.5, sino 0
    prediccion_binaria = 1 if prob_misogino > 0.5 else 0

    y_true.append(true_label)
    y_pred.append(prediccion_binaria)

    # Guardamos para el Ensemble Multimodal
    resultados_para_csv.append({
        "id_EXIST": id_vid,
        "prob_misogino_video": prob_misogino,
        "prediccion_binaria_video": prediccion_binaria,
        "label_real": true_label
    })

  2%|▏         | 8/502 [00:04<02:39,  3.09it/s]


⚠️ Error procesando /content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/videos/6927331134334373126.mp4: [18:53:22] /github/workspace/src/video/ffmpeg/threaded_decoder.cc:292: [18:53:22] /github/workspace/src/video/ffmpeg/filter_graph.cc:100: Check failed: av_buffersrc_add_frame_flags(buffersrc_ctx_, frame, AV_BUFFERSRC_FLAG_KEEP_REF) >= 0 (-22 vs. 0) Error while feeding the filter graph


 16%|█▌        | 78/502 [01:16<08:46,  1.24s/it]


⚠️ Error procesando /content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/videos/6993013215147855109.mp4: [18:54:34] /github/workspace/src/video/video_reader.cc:486: Error: av_read_frame failed with 1094995529


 28%|██▊       | 142/502 [02:46<07:13,  1.20s/it]

...

 94%|█████████▍| 471/502 [11:02<00:42,  1.36s/it]


⚠️ Error procesando /content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/videos/6957860609790676230.mp4: [19:04:20] /github/workspace/src/video/video_reader.cc:486: Error: av_read_frame failed with 1094995529


 98%|█████████▊| 494/502 [11:33<00:10,  1.28s/it]


⚠️ Error procesando /content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/videos/6985683837988752646.mp4: [19:04:51] /github/workspace/src/video/ffmpeg/threaded_decoder.cc:292: [19:04:51] /github/workspace/src/video/ffmpeg/filter_graph.cc:100: Check failed: av_buffersrc_add_frame_flags(buffersrc_ctx_, frame, AV_BUFFERSRC_FLAG_KEEP_REF) >= 0 (-22 vs. 0) Error while feeding the filter graph


100%|██████████| 502/502 [11:46<00:00,  1.41s/it]


## 4. Guardado del csv de predicción y muestreo de métricas

In [ ]:
# Guardamos el CSV
os.makedirs(os.path.dirname(CSV_SALIDA), exist_ok=True)
pd.DataFrame(resultados_para_csv).to_csv(CSV_SALIDA, index=False)
print(f"\n✅ ¡CSV de VideoMAE guardado para el Ensemble en: {CSV_SALIDA}!")


✅ ¡CSV de VideoMAE guardado para el Ensemble en: /content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/Vision/predicciones_videomae_test.csv!


In [ ]:
print("\n" + "="*50)
print("🏆 RESULTADOS TEST ESTÁTICO: VideoMAE (Análisis Temporal)")
print("="*50)
print(f"F1-Score (Macro): {f1_score(y_true, y_pred, average='macro'):.4f}")
print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
print("\nMatriz de Confusión:\n", confusion_matrix(y_true, y_pred))
print("\nClassification Report:\n", classification_report(y_true, y_pred, target_names=["No Misógino", "Misógino"]))


🏆 RESULTADOS TEST ESTÁTICO: VideoMAE (Análisis Temporal)
F1-Score (Macro): 0.4887
Accuracy: 0.5279

Matriz de Confusión:
 [[202  59]
 [178  63]]

Classification Report:
               precision    recall  f1-score   support

 No Misógino       0.53      0.77      0.63       261
    Misógino       0.52      0.26      0.35       241

    accuracy                           0.53       502
   macro avg       0.52      0.52      0.49       502
weighted avg       0.52      0.53      0.49       502



# Entrenamiento con el 100% de los datos

In [4]:
import os
import torch
import numpy as np
import pandas as pd
from decord import VideoReader, cpu
import decord
from torch.utils.data import Dataset
from transformers import (
    VideoMAEImageProcessor,
    VideoMAEForVideoClassification,
    TrainingArguments,
    Trainer
)

In [5]:
# Silenciamos los logs de decord
decord.bridge.set_bridge('torch')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. Configuración y rutas

In [6]:
MODEL_CHECKPOINT = "MCG-NJU/videomae-base"

# Ruta base donde están los archivos .mp4
RUTA_BASE_VIDEOS = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/"
# El CSV con el 100% de los datos
CSV_TODO_EL_DATASET = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/EXIST2025_training_3_1.csv"

# Ruta de salida para el modelo final
OUTPUT_DIR = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/SubidasCompeticion/videomae-base"
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 2. Carga del dataset

In [7]:
print("Cargando el 100% del dataset maestro...")
df_completo = pd.read_csv(CSV_TODO_EL_DATASET)

if "Unnamed: 0" in df_completo.columns:
    df_completo.drop(columns=["Unnamed: 0"], inplace=True)
if "label_task_3_1_merged" in df_completo.columns:
    df_completo.rename(columns={"label_task_3_1_merged": "label"}, inplace=True)
df_completo["label"] = df_completo["label"].astype(int)

Cargando el 100% del dataset maestro...


In [8]:
def fix_video_paths(df, base_path):
    columna = 'path_video' if 'path_video' in df.columns else 'id_EXIST'
    rutas = []
    for val in df[columna]:
        val_str = str(val)
        if not val_str.endswith(".mp4"):
            val_str += ".mp4"

        # Lógica robusta para evitar "/videos/videos/"
        if "videos/" in val_str:
            ruta_completa = os.path.join(base_path, val_str)
        else:
            ruta_completa = os.path.join(base_path, "videos", val_str)
        rutas.append(ruta_completa)

    df['ruta_absoluta'] = rutas
    return df

df_completo = fix_video_paths(df_completo.copy(), RUTA_BASE_VIDEOS)
print(f"Total de vídeos para entrenar: {len(df_completo)}")

Total de vídeos para entrenar: 2508


## 3. Extracción de Frames y Dataset de PyTorch

In [9]:
print(f"Cargando procesador {MODEL_CHECKPOINT}...")
processor = VideoMAEImageProcessor.from_pretrained(MODEL_CHECKPOINT)

Cargando procesador MCG-NJU/videomae-base...


preprocessor_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

In [10]:
def sample_frame_indices(clip_len, total_frames):
    if total_frames <= clip_len:
        return np.linspace(0, total_frames - 1, num=clip_len, dtype=int).tolist()
    else:
        return np.linspace(0, total_frames - 1, num=clip_len, dtype=int).tolist()

In [11]:
class VideoClassificationDataset(Dataset):
    def __init__(self, df, processor, clip_len=16):
        self.rutas = df['ruta_absoluta'].tolist()
        self.etiquetas = df['label'].tolist()
        self.processor = processor
        self.clip_len = clip_len

    def __len__(self):
        return len(self.rutas)

    def __getitem__(self, idx):
        ruta_video = self.rutas[idx]
        etiqueta = self.etiquetas[idx]

        try:
            vr = VideoReader(ruta_video, ctx=cpu(0))
            total_frames = len(vr)
            frame_indices = sample_frame_indices(self.clip_len, total_frames)
            frames = vr.get_batch(frame_indices).numpy()

            inputs = self.processor(list(frames), return_tensors="pt")
            pixel_values = inputs["pixel_values"][0]

        except Exception as e:
            # Salvavidas para vídeos corruptos
            pixel_values = torch.zeros((self.clip_len, 3, 224, 224))
            etiqueta = 0

        return {"pixel_values": pixel_values, "label": etiqueta}

train_dataset = VideoClassificationDataset(df_completo, processor, clip_len=16)

In [12]:
def collate_fn(batch):
    return {
        'pixel_values': torch.stack([x['pixel_values'] for x in batch]),
        'labels': torch.tensor([x['label'] for x in batch])
    }

## 4. Inicialización del modelo VideoMAE

In [13]:
print("Cargando modelo VideoMAEForVideoClassification...")
id2label = {0: "No Misógino", 1: "Misógino"}
label2id = {"No Misógino": 0, "Misógino": 1}

model = VideoMAEForVideoClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

Cargando modelo VideoMAEForVideoClassification...


config.json:   0%|          | 0.00/725 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  377MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/160 [00:00<?, ?it/s]

[transformers] VideoMAEForVideoClassification LOAD REPORT from: MCG-NJU/videomae-base
Key                                                                  | Status     | 
---------------------------------------------------------------------+------------+-
decoder.decoder_layers.{0, 1, 2, 3}.intermediate.dense.bias          | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.value.weight | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.key.weight   | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.output.dense.weight    | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.q_bias       | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.layernorm_after.weight           | UNEXPECTED | 
videomae.encoder.layer.{0...11}.attention.attention.q_bias           | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.v_bias       | UNEXPECTED | 
decoder.norm.weight                                             

## 5. Entrenamiento Ciego (con el 100% de los datos)

In [15]:
batch_size = 4

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    remove_unused_columns=False, # ¡VITAL EN VÍDEO!
    learning_rate=5e-5,
    per_device_train_batch_size=batch_size,
    gradient_accumulation_steps=2,
    num_train_epochs=3,          # 3 épocas (balance ideal sin validación)
    weight_decay=0.01,
    fp16=True,
    logging_strategy="epoch",
    eval_strategy="no",
    save_strategy="no",
    report_to="none",
    dataloader_num_workers=2,
    dataloader_pin_memory=True
)

In [16]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=collate_fn
)

In [17]:
print("\n🚀 Lanzando el entrenamiento de VideoMAE al 100%...")
trainer.train()

print("\n💾 Guardando el modelo definitivo...")
trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)

print("✅ ¡Entrenamiento completado y guardado con éxito!")


🚀 Lanzando el entrenamiento de VideoMAE al 100%...


Step,Training Loss
314,0.702705
628,0.668635
942,0.545889



💾 Guardando el modelo definitivo...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ ¡Entrenamiento completado y guardado con éxito!


# Predicción contra el Test de la competición

In [18]:
import pandas as pd
import numpy as np
import torch
import json
import os
from tqdm import tqdm
from decord import VideoReader, cpu
import decord
from transformers import VideoMAEImageProcessor, VideoMAEForVideoClassification

In [19]:
decord.bridge.set_bridge('torch')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. Configuración y rutas

In [31]:
RUTA_MODELO_FINAL = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/SubidasCompeticion/videomae-base"

# Ruta al CSV estático de Test (El que usabas en el cuaderno)
RUTA_TEST_JSON = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/test/EXIST2025_test_clean.json"
RUTA_BASE_VIDEOS = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/test/"

RUTA_SUBMISSION_JSON = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/SubidasCompeticion/videomae_test_oficial.json"

## 2. Carga del test y procesador

In [26]:
print("Cargando el test oficial limpio...")
df_test = pd.read_json(RUTA_TEST_JSON, orient='index')
if "Unnamed: 0" in df_test.columns:
    df_test.drop(columns=["Unnamed: 0"], inplace=True)

Cargando el test oficial limpio...


In [27]:
print(f"Cargando Procesador y Modelo VideoMAE desde {RUTA_MODELO_FINAL}...")
processor = VideoMAEImageProcessor.from_pretrained(RUTA_MODELO_FINAL)

Cargando Procesador y Modelo VideoMAE desde /content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/SubidasCompeticion/videomae-base...


In [28]:
# Nota: Forzamos la carga directa a device (GPU) como vimos en Mistral para evitar fallos
model = VideoMAEForVideoClassification.from_pretrained(RUTA_MODELO_FINAL).to(device)
model.eval()

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

VideoMAEForVideoClassification(
  (videomae): VideoMAEModel(
    (embeddings): VideoMAEEmbeddings(
      (patch_embeddings): VideoMAEPatchEmbeddings(
        (projection): Conv3d(3, 768, kernel_size=(2, 16, 16), stride=(2, 16, 16))
      )
    )
    (encoder): VideoMAEEncoder(
      (layer): ModuleList(
        (0-11): 12 x VideoMAELayer(
          (attention): VideoMAEAttention(
            (attention): VideoMAESelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
            )
            (output): VideoMAESelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): VideoMAEIntermediate(
            (dense): Linear(in_features=768, out_features=3072, bias=True)
       

In [29]:
def sample_frame_indices(clip_len, total_frames):
    if total_frames <= clip_len:
        return np.linspace(0, total_frames - 1, num=clip_len, dtype=int).tolist()
    else:
        return np.linspace(0, total_frames - 1, num=clip_len, dtype=int).tolist()

## 3. Inferencia manual por video

In [32]:
print(f"Realizando predicciones sobre {len(df_test)} vídeos...")
predicciones = []

for index, row in tqdm(df_test.iterrows(), total=len(df_test)):
    id_vid = row["id_EXIST"]
    val_str = str(row.get('path_video', id_vid))

    if not val_str.endswith(".mp4"):
        val_str += ".mp4"
    if "videos/" in val_str:
        ruta_video = os.path.join(RUTA_BASE_VIDEOS, val_str)
    else:
        ruta_video = os.path.join(RUTA_BASE_VIDEOS, "videos", val_str)

    prob_misogino = 0.5 # Neutro si falla

    try:
        vr = VideoReader(ruta_video, ctx=cpu(0))
        frame_indices = sample_frame_indices(16, len(vr))
        frames = vr.get_batch(frame_indices).asnumpy() # .asnumpy() en lugar de .numpy() por compatibilidad

        inputs = processor(list(frames), return_tensors="pt")
        pixel_values = inputs["pixel_values"].to(device)

        with torch.no_grad():
            outputs = model(pixel_values=pixel_values)
            probs = torch.nn.functional.softmax(outputs.logits, dim=-1)[0]
            prob_misogino = probs[1].item()

    except Exception as e:
        # Silenciamos el print para que no te ensucie la barra de progreso si hay muchos rotos
        pass

    pred_binaria = 1 if prob_misogino > 0.5 else 0
    predicciones.append(pred_binaria)

df_test["predicted_label"] = predicciones

Realizando predicciones sobre 674 vídeos...


100%|██████████| 674/674 [11:25<00:00,  1.02s/it]


## 4. Formateo al estándar PyEvall

In [33]:
print("Generando archivo de sumisión en formato PyEvALL...")
output_json = []
for index, row in df_test.iterrows():
    entry = {
        "test_case": "EXIST2025",
        "id": str(row["id_EXIST"]),
        "value": "YES" if int(row["predicted_label"]) == 1 else "NO"
    }
    output_json.append(entry)

os.makedirs(os.path.dirname(RUTA_SUBMISSION_JSON), exist_ok=True)
with open(RUTA_SUBMISSION_JSON, "w", encoding="utf-8") as f:
    json.dump(output_json, f, indent=2)

print(f"✅ ¡Completado! Archivo guardado en: {RUTA_SUBMISSION_JSON}")

Generando archivo de sumisión en formato PyEvALL...
✅ ¡Completado! Archivo guardado en: /content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/SubidasCompeticion/videomae_test_oficial.json
